In [1]:
import pandas as pd

In [ ]:
path_1 = r"C:\Users\vaibh\OneDrive\Desktop\multi_strategy_generator\results\strategy_results_1D_top500_lookback_commission_standard_new_data_ethusdt_updated_validated.csv"
path_2 = r"C:\Users\vaibh\OneDrive\Desktop\multi_strategy_generator\results\strategy_results_15m_top500_lookback_commission_standard_new_data_btcusdt_updated_validated.csv"
path_3 = r"C:\Users\vaibh\OneDrive\Desktop\multi_strategy_generator\results\strategy_results_60m_top500_lookback_commission_standard_new_data_btcusdt_updated_validated.csv"
df_1 = pd.read_csv(path_1) # one day data
df_2 = pd.read_csv(path_2) # 15 min data
df_3 = pd.read_csv(path_3) # 60 min data

df_1.columns

Index(['id', 'direction', 'signals', 'n_signals', 'tp', 'sl', 'train_return',
       'train_sharpe', 'train_drawdown', 'train_trades', 'train_winrate',
       'score', 'test_return', 'test_sharpe', 'test_drawdown', 'test_trades',
       'test_winrate', 'test_score'],
      dtype='object')

In [24]:
res = df_1[df_1['test_return']>0]
print(len(res))
# import numpy as np 
# print(np.argmax(df_1['test_return']))
r = (res[res['n_signals']<4])
print(r[['tp','sl','n_signals','signals']])

88
           tp        sl  n_signals       signals
18   0.058926  0.018536          3   s43|s73|s94
34   0.063002  0.018729          3   s68|s10|s94
35   0.071859  0.019409          3   s95|s94|s91
39   0.070451  0.017969          3   s44|s87|s21
45   0.071913  0.016390          3    s21|s8|s95
46   0.040559  0.014966          3   s87|s54|s74
59   0.055859  0.019976          3   s94|s43|s68
65   0.064496  0.019058          3   s71|s19|s86
92   0.074658  0.017095          3   s86|s19|s26
98   0.078244  0.014397          3   s21|s88|s26
107  0.046338  0.017846          3  s100|s18|s26
140  0.079881  0.013820          3   s35|s21|s68
171  0.066521  0.014725          3   s11|s21|s68
220  0.048851  0.019325          3   s82|s54|s87
235  0.067465  0.010032          3   s56|s86|s26
254  0.046135  0.019268          3   s54|s47|s69
263  0.073192  0.018016          3   s30|s63|s94
270  0.067975  0.017375          3   s93|s21|s88
287  0.067299  0.014787          3  s29|s28|s101
288  0.038026  0.

In [26]:
res = df_2[df_2['test_return']>0]
print(len(res))
# import numpy as np 
# print(np.argmax(df_2['test_return']))
r = (res[res['n_signals']<5])
print(r[['tp','sl','n_signals','signals']])

6
           tp        sl  n_signals           signals
27   0.025669  0.013325          4  s52|s28|s101|s66
72   0.025087  0.010870          4   s29|s86|s62|s81
96   0.022526  0.013488          4  s52|s98|s16|s100
119  0.025870  0.015826          4  s29|s28|s100|s18
188  0.027219  0.015737          4   s90|s80|s29|s62


In [27]:
res = df_3[df_3['test_return']>0]
print(len(res))
# import numpy as np 
# print(np.argmax(df_3['test_return']))
r = (res[res['n_signals']<4])
print(r[['tp','sl','n_signals','signals']])

190
           tp        sl  n_signals       signals
37   0.041691  0.015468          2      s23|s101
41   0.079094  0.018995          3   s79|s83|s54
48   0.052695  0.019679          2       s13|s81
50   0.053530  0.016605          3   s36|s63|s83
65   0.055368  0.019594          3   s81|s13|s67
85   0.027530  0.013771          3   s13|s66|s89
87   0.026927  0.017254          3   s89|s13|s26
92   0.076267  0.019698          2       s73|s91
95   0.051442  0.015416          3   s91|s73|s92
112  0.061885  0.019345          2       s23|s18
148  0.045611  0.015204          3   s25|s23|s98
160  0.039843  0.018543          3   s73|s63|s91
194  0.071453  0.014540          3   s76|s13|s81
202  0.059876  0.015750          2       s83|s36
248  0.069325  0.019219          3  s100|s62|s13
253  0.070022  0.011693          3   s59|s92|s21
268  0.035280  0.016710          3   s59|s55|s73
355  0.054356  0.013776          3   s95|s87|s48


In [8]:
import numpy as np
import pandas as pd
import ta
from backtesting import Backtest, Strategy

# ─────────────────────────────────────────────────────────────────────────────
#  SMART MONEY CONCEPTS
# ─────────────────────────────────────────────────────────────────────────────

def smart_money_concepts(df, swing_length=50, internal_length=5, eql_length=3, eql_threshold=0.1):
    high  = df["high"].values
    low   = df["low"].values
    close = df["close"].values
    n     = len(close)

    prev_c = np.roll(close, 1); prev_c[0] = close[0]
    tr     = np.maximum(high - low, np.maximum(np.abs(high - prev_c), np.abs(low - prev_c)))
    atr200 = pd.Series(tr).ewm(alpha=1/200, adjust=False).mean().values

    def get_legs(size):
        legs = np.full(n, -1, dtype=int)
        for i in range(size, n):
            win_h = high[max(0, i - size): i]
            win_l = low [max(0, i - size): i]
            if len(win_h) == 0:
                continue
            if high[i] > win_h.max():   legs[i] = 0
            elif low[i] < win_l.min():  legs[i] = 1
            else:                       legs[i] = legs[i - 1] if i > 0 else -1
        return legs

    def detect_structure(size):
        legs       = get_legs(size)
        bull_bos   = np.zeros(n, dtype=int)
        bull_choch = np.zeros(n, dtype=int)
        bear_bos   = np.zeros(n, dtype=int)
        bear_choch = np.zeros(n, dtype=int)
        cur_high   = np.nan; cur_low = np.nan
        high_crossed = False; low_crossed = False
        trend = 0; prev_leg = -1

        for i in range(size, n):
            cur_leg = legs[i]
            if cur_leg != prev_leg and prev_leg != -1:
                pbar = max(0, i - size)
                if cur_leg == 0: cur_high = high[pbar]; high_crossed = False
                if cur_leg == 1: cur_low  = low[pbar];  low_crossed  = False

            if not np.isnan(cur_high) and not high_crossed:
                pc = close[i-1] if i > 0 else close[i]
                if pc <= cur_high < close[i]:
                    if trend <= 0: bull_choch[i] = 1
                    else:          bull_bos[i]   = 1
                    high_crossed = True; trend = 1

            if not np.isnan(cur_low) and not low_crossed:
                pc = close[i-1] if i > 0 else close[i]
                if pc >= cur_low > close[i]:
                    if trend >= 0: bear_choch[i] = 1
                    else:          bear_bos[i]   = 1
                    low_crossed = True; trend = -1

            prev_leg = cur_leg

        return bull_bos, bull_choch, bear_bos, bear_choch

    (sw_bb, sw_bc, sw_rb, sw_rc)   = detect_structure(swing_length)
    (int_bb, int_bc, int_rb, int_rc) = detect_structure(internal_length)

    eql_legs = get_legs(eql_length)
    eq_high  = np.zeros(n, dtype=int)
    eq_low   = np.zeros(n, dtype=int)
    prev_ph  = np.nan; prev_pl = np.nan; prev_eleg = -1

    for i in range(eql_length, n):
        cur_eleg = eql_legs[i]
        if cur_eleg != prev_eleg and prev_eleg != -1:
            pbar = max(0, i - eql_length)
            if cur_eleg == 0:
                cur_ph = high[pbar]
                if not np.isnan(prev_ph) and abs(cur_ph - prev_ph) < eql_threshold * atr200[i]:
                    eq_high[i] = 1
                prev_ph = cur_ph
            if cur_eleg == 1:
                cur_pl = low[pbar]
                if not np.isnan(prev_pl) and abs(cur_pl - prev_pl) < eql_threshold * atr200[i]:
                    eq_low[i] = 1
                prev_pl = cur_pl
        prev_eleg = cur_eleg

    def rolling_trend(bb, bc, rb, rc):
        t_arr = np.zeros(n, dtype=int); t = 0
        for i in range(n):
            if bb[i] or bc[i]:   t = 1
            elif rb[i] or rc[i]: t = -1
            t_arr[i] = t
        return t_arr

    swing_trend    = rolling_trend(sw_bb,  sw_bc,  sw_rb,  sw_rc)
    internal_trend = rolling_trend(int_bb, int_bc, int_rb, int_rc)

    idx = df.index
    df["smc_int_is_bull"] = (pd.Series(internal_trend, index=idx) == 1).astype(int)
    df["smc_int_is_bear"] = (pd.Series(internal_trend, index=idx) == -1).astype(int)
    df["smc_sw_is_bull"]  = (pd.Series(swing_trend,    index=idx) == 1).astype(int)
    df["smc_sw_is_bear"]  = (pd.Series(swing_trend,    index=idx) == -1).astype(int)
    return df


# ─────────────────────────────────────────────────────────────────────────────
#  PREPARE DATA
# ─────────────────────────────────────────────────────────────────────────────

# path = r"C:\Users\vaibh\OneDrive\Desktop\multi_strategy_generator\data\ethusd_60m.csv"
path = r"C:\Users\vaibh\OneDrive\Desktop\multi_strategy_generator\data\btcusd_60m.csv"
df = pd.read_csv(path)

# convert timestamp (ms) → datetime
df["time"] = pd.to_datetime(df["time"], unit="ms")
# set index
df.set_index("time", inplace=True)
df.columns = [c.lower() for c in df.columns]

# Compute indicators before passing to Backtest
df["ultimate_osc"] = ta.momentum.ultimate_oscillator(df["high"], df["low"], df["close"], 7, 14, 28)
df = smart_money_concepts(df)
df.dropna(inplace=True)

# backtesting.py requires Title-case OHLCV columns
df = df.rename(columns={"open": "Open", "high": "High", "low": "Low",
                         "close": "Close", "volume": "Volume"})


# ─────────────────────────────────────────────────────────────────────────────
#  STRATEGY  s23 / s101
#
#  LONG  : UO < 20  AND smc_int_is_bear == 1
#  SHORT : UO > 80  AND smc_int_is_bull == 1
#
#  TP = 4.1691 %   SL = 1.5468 %
# ─────────────────────────────────────────────────────────────────────────────

class SMC_S23_S101(Strategy):
    tp_pct = 0.041691
    sl_pct = 0.015468

    def init(self):
        self.uo       = self.I(lambda: self.data.ultimate_osc,   name="UO")
        self.int_bear = self.I(lambda: self.data.smc_int_is_bear, name="IntBear")

    def next(self):
        price = self.data.Close[-1]

        # SHORT: overbought UO + internal bullish structure (mean-reversion) - short only strategy
        if self.uo[-1] < 30 and self.int_bear[-1] == 1:
            if not self.position.is_short:
                self.sell(
                    tp = price * (1 - self.tp_pct),
                    sl = price * (1 + self.sl_pct),
                )


# ─────────────────────────────────────────────────────────────────────────────
#  RUN BACKTEST
# ─────────────────────────────────────────────────────────────────────────────

bt = Backtest(
    df,
    SMC_S23_S101,
    cash             = 1_000_000,
    commission       = 0.001,   # 0.1% per side (typical crypto)
    exclusive_orders = True,    # close existing before opening opposite
)

stats = bt.run()
print(stats)

bt.plot()   # opens interactive HTML chart in browser


# ─────────────────────────────────────────────────────────────────────────────
#  SAMBO OPTIMISATION  (commented — uncomment to use)
#
#  bt.optimize() with method="sambo" runs Bayesian / Sequential Model-Based
#  Optimisation instead of a brute-force grid search.
#  Install dependency:  pip install sambo
# ─────────────────────────────────────────────────────────────────────────────

# stats_opt, heatmap = bt.optimize(
#     tp_pct        = [round(x, 4) for x in np.arange(0.020, 0.080, 0.005)],
#     sl_pct        = [round(x, 4) for x in np.arange(0.008, 0.030, 0.003)],
#     maximize      = "Equity Final [$]",
#     method        = "sambo",        # <-- Bayesian search
#     max_tries     = 200,            # evaluations budget
#     constraint    = lambda p: p.tp_pct > p.sl_pct,
#     return_heatmap= True,
# )
# print(stats_opt._strategy)
# print(stats_opt)

Backtest.run:   0%|          | 0/17460 [00:00<?, ?bar/s]

Start                     2024-04-10 17:00:00
End                       2026-04-09 13:00:00
Duration                    728 days 20:00:00
Exposure Time [%]                    20.25084
Equity Final [$]                1676223.64552
Equity Peak [$]                 1826025.39616
Commissions [$]                  258301.70282
Return [%]                           67.62236
Buy & Hold Return [%]                 2.02148
Return (Ann.) [%]                    29.46906
Volatility (Ann.) [%]                23.64424
CAGR [%]                             29.52259
Sharpe Ratio                          1.24635
Sortino Ratio                         2.90689
Calmar Ratio                          2.71624
Alpha [%]                            67.92646
Beta                                 -0.15043
Max. Drawdown [%]                   -10.84922
Avg. Drawdown [%]                     -2.0563
Max. Drawdown Duration      159 days 19:00:00
Avg. Drawdown Duration        9 days 16:00:00
# Trades                          

c:\Users\vaibh\anaconda3\envs\new_env\lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '2h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\vaibh\anaconda3\envs\new_env\lib\site-packages\backtesting\_plotting.py:709: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
c:\Users\vaibh\anaconda3\envs\new_env\lib\site-packages\backtesting\_plotting.py:709: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(


GridPlot(id='p1415', ...)

In [6]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
path = r"C:\Users\vaibh\OneDrive\Desktop\multi_strategy_generator\results\strategy_results_60m_top500_lookback_commission_standard_new_data_btcusdt_updated_validated.csv"

MIN_SUPPORT = 0.07      # how frequently a combination must appear
MIN_LIFT = 0.8          # strength of association rule
TOP_PERCENT = 0.5       # use top 50% strategies based on performance


# ─────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────
df = pd.read_csv(path)

# Keep only top-performing strategies (important to avoid learning noise)
df = df.sort_values("test_score", ascending=False)
df = df.head(int(len(df) * TOP_PERCENT))


# ─────────────────────────────────────────────────────────────
# PREPARE TRANSACTIONS
# ─────────────────────────────────────────────────────────────
# Convert "s8|s21|s33" → ["s8", "s21", "s33"]
df["signal_list"] = df["signals"].apply(lambda x: x.split("|"))

transactions = df["signal_list"].tolist()


# ─────────────────────────────────────────────────────────────
# ENCODE DATA (required for Apriori)
# ─────────────────────────────────────────────────────────────
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)


# ─────────────────────────────────────────────────────────────
# APRIORI ALGORITHM
# ─────────────────────────────────────────────────────────────
"""
Apriori finds frequently occurring combinations of items.

In your case:
- Items = signals (s21, s33, etc.)
- Goal = find combinations of signals that appear often in good strategies

Example output:
{ s21, s33 } → appears in 40% of top strategies
"""

frequent_itemsets = apriori(
    df_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

print("\n=== FREQUENT SIGNAL COMBINATIONS ===")
print(frequent_itemsets.sort_values("support", ascending=False))


# ─────────────────────────────────────────────────────────────
# ASSOCIATION RULES
# ─────────────────────────────────────────────────────────────
"""
Association rules tell you relationships like:

{s21} → {s33}

Meaning:
If s21 is present, s33 is likely also present

Metrics:
- confidence: probability of rule being true
- lift: strength of relationship (>1 is good)
"""

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=MIN_LIFT
)

print("\n=== ASSOCIATION RULES ===")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]]
      .sort_values("confidence", ascending=False))


# ─────────────────────────────────────────────────────────────
# SAVE RESULTS
# ─────────────────────────────────────────────────────────────
frequent_itemsets.to_csv("frequent_itemsets.csv", index=False)
rules.to_csv("association_rules.csv", index=False)

print("\nSaved results to:")
print(" - frequent_itemsets.csv")
print(" - association_rules.csv")


=== FREQUENT SIGNAL COMBINATIONS ===
    support     itemsets
5     0.312        (s23)
0     0.212       (s100)
16    0.196        (s81)
1     0.196        (s13)
4     0.192        (s22)
12    0.164        (s62)
10    0.156        (s51)
13    0.136        (s66)
6     0.124        (s25)
15    0.124        (s76)
3     0.120        (s20)
8     0.116        (s29)
14    0.116        (s67)
19    0.112        (s89)
22    0.104  (s100, s62)
18    0.100        (s86)
21    0.088        (s98)
9     0.084        (s50)
7     0.080        (s28)
2     0.080        (s16)
23    0.080   (s81, s13)
24    0.076   (s23, s25)
11    0.072        (s61)
17    0.072        (s83)
20    0.072        (s97)

=== ASSOCIATION RULES ===
  antecedents consequents  support  confidence      lift
1       (s62)      (s100)    0.104    0.634146  2.991256
5       (s25)       (s23)    0.076    0.612903  1.964433
0      (s100)       (s62)    0.104    0.490566  2.991256
2       (s81)       (s13)    0.080    0.408163  2.082466
